# النظام الهجين — SAM2 + AOT-GAN
> رسم على التلف ← SAM2 يستخرج القناع ← AOT-GAN يرمم الصورة

In [ ]:
# ════════════════════════════════════════
# الخلية 1 — إعداد البيئة
# ════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import sys, os, subprocess, torch

ROOT      = '/content/drive/MyDrive/hybrid_project'
SAM2_ROOT = f'{ROOT}/modules/SAM2'
AOT_SRC   = f'{ROOT}/modules/AOT_GAN/src'

# تثبيت المكتبات
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'iopath', 'hydra-core'], check=True)

# إضافة المسارات
for p in [SAM2_ROOT, AOT_SRC, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# فحص الإعدادات
from pipeline.config import print_config
print_config()

# تحميل الأنبوب
from pipeline.run import HybridPipeline
pipeline = HybridPipeline()
print('\n✅ النظام الهجين جاهز')

In [ ]:
# ════════════════════════════════════════
# الخلية 2 — رفع الصورة يدوياً
# ════════════════════════════════════════
from google.colab import files
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import io

print('📂 اختر صورة من جهازك:')
uploaded = files.upload()

fname    = list(uploaded.keys())[0]
img_data = uploaded[fname]

image_np = np.array(
    Image.open(io.BytesIO(img_data)).convert('RGB').resize((512, 512))
)

# تحميلها في pipeline
pipeline.image_np = image_np
pipeline.img_path = fname
pipeline.sam2.set_image(image_np)

H, W = image_np.shape[:2]
print(f'✅ الصورة محملة: {fname} — {W}×{H}')

plt.figure(figsize=(5, 5))
plt.imshow(image_np)
plt.axis('off')
plt.title(fname)
plt.show()

In [ ]:
# ════════════════════════════════════════
# الخلية 3 — واجهة الرسم
# ارسم على منطقة التلف ثم اضغط تأكيد
# ════════════════════════════════════════
import base64, io as _io
from PIL import Image
from IPython.display import display, HTML
from google.colab import output

H, W = image_np.shape[:2]

buf = _io.BytesIO()
Image.fromarray(image_np).save(buf, format='PNG')
b64 = base64.b64encode(buf.getvalue()).decode()

def on_mask_received(draw_b64):
    try:
        result = pipeline.process_drawn_mask(draw_b64)
        pipeline.show_result(result)
    except Exception as e:
        print(f'❌ خطأ: {e}')
        import traceback; traceback.print_exc()

output.register_callback('process_mask', on_mask_received)

display(HTML(f'''
<div style="position:relative;display:inline-block;
            border:2px solid #555;border-radius:8px;overflow:hidden">
  <img id="bg" src="data:image/png;base64,{b64}"
       style="display:block;max-width:512px;width:100%">
  <canvas id="cv"
       style="position:absolute;top:0;left:0;
              width:100%;height:100%;opacity:0.55;cursor:crosshair">
  </canvas>
</div>
<div style="margin-top:8px;display:flex;gap:10px;
            align-items:center;flex-wrap:wrap">
  <label>حجم الفرشاة:
    <input id="brush" type="range" min="5" max="60" value="20"
           style="width:120px">
  </label>
  <button onclick="clearCanvas()"
    style="padding:6px 14px;background:#e74c3c;color:#fff;
           border:none;border-radius:6px;cursor:pointer">مسح</button>
  <button onclick="exportMask()"
    style="padding:6px 14px;background:#27ae60;color:#fff;
           border:none;border-radius:6px;cursor:pointer">
    ✅ تأكيد ← SAM2 + AOT-GAN
  </button>
  <span id="status" style="color:#888;font-size:13px"></span>
</div>
<script>
const bg=document.getElementById("bg"),cv=document.getElementById("cv"),
      ctx=cv.getContext("2d");
function init(){{cv.width=bg.clientWidth;cv.height=bg.clientHeight;
  ctx.fillStyle="black";ctx.fillRect(0,0,cv.width,cv.height);}}
bg.complete?init():(bg.onload=init);
let down=false;
cv.onmousedown=e=>{{down=true;draw(e);}};
cv.onmousemove=e=>{{if(down)draw(e);}};
cv.onmouseup=()=>down=false;
cv.onmouseleave=()=>down=false;
function draw(e){{
  const r=parseInt(document.getElementById("brush").value);
  const rec=cv.getBoundingClientRect();
  ctx.beginPath();
  ctx.arc(e.clientX-rec.left,e.clientY-rec.top,r,0,Math.PI*2);
  ctx.fillStyle="white";ctx.fill();}}
function clearCanvas(){{
  ctx.fillStyle="black";ctx.fillRect(0,0,cv.width,cv.height);
  document.getElementById("status").textContent="";}}
function exportMask(){{
  document.getElementById("status").textContent="⏳ جاري المعالجة...";
  const tmp=document.createElement("canvas");
  tmp.width={W};tmp.height={H};
  tmp.getContext("2d").drawImage(cv,0,0,{W},{H});
  google.colab.kernel.invokeFunction(
    "process_mask",[tmp.toDataURL("image/png").split(",")[1]],{{}});
  document.getElementById("status").textContent="✅ تم الإرسال — انتظر النتيجة";}}
</script>
'''))

In [ ]:
# ════════════════════════════════════════
# خلية مساعدة — تحديث الملفات بدون restart
# شغّلها إذا عدّلت أي ملف في pipeline/
# ════════════════════════════════════════
import importlib
import pipeline.config, pipeline.selector, pipeline.segmentation
import pipeline.inpainting, pipeline.run

for mod in [pipeline.config, pipeline.selector,
            pipeline.segmentation, pipeline.inpainting, pipeline.run]:
    importlib.reload(mod)

print('✅ تم تحديث جميع الوحدات')